# Efficient trusted low-rank detector

## Method

### Evidence and trusted-model assessment

Each example contains the four most recent user-visible context messages and the final assistant response. System and developer messages are omitted, and long text is clipped from the middle. This evidence is placed in a fixed truthfulness-assessment prompt for `Qwen/Qwen3.5-9B`. The detector reads an activation without generating a response.

### Activation features

At layer 23, the detector mean-pools the final 16 and final 24 prompt tokens. The feature concatenates the final-16 mean with the difference between the final-16 and final-24 means.

### Low-rank readout and training

Training uses 2,176 public examples from instructed and varied deception datasets. Each scenario and source group contributes the same number of honest and deceptive examples. Nine groups come from the organizer development datasets and eight from public `liars-bench-expanded` metadata. After standardization, one deceptive-minus-honest mean direction is computed for each group. An L2-regularized logistic classifier (`C=0.001`) is fitted on the resulting 17-dimensional representation, then collapsed into a single projection in the original feature space.

The stored threshold maximizes balanced accuracy on the training sample. Inference subtracts the threshold logit before the sigmoid, so `0.5` remains the decision boundary. The evidence construction, prompt, layer, token windows, projection, and threshold are fixed across evaluation datasets. `train_trusted_activation_probe.py` rebuilds the artifact, `training_datasets.yaml` lists the organizer datasets, and `liars_bench_data.py` loads the additional public metadata and labels.

### Efficient inference

Prompts are tokenized once, sorted by length, and packed into batches of at most 32 rows and 12,000 padded tokens. All batches run in one remote NNsight session. Pooling, standardization, projection, threshold adjustment, and the sigmoid run remotely, so only one score per example is downloaded. Execution stops after layer 23.

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd

DATASET_NAME = os.environ["DATASET_NAME"]
LIMIT = int(os.environ["ALETHEIA_LIMIT"]) if os.environ.get("ALETHEIA_LIMIT") else None
sys.path.insert(0, "submission")
from util import load_examples, write_submission
from activation_detector import load_probe, score_examples

## Fixed assessment prompt

The prompt is defined once in `activation_detector.py` and used for both training and inference. The detector reads layer 23 and does not generate the requested continuation.

In [ ]:
PROBE_PATH = Path("submission/trusted_activation_probe.npz")
BATCH_SIZE = 32

In [ ]:
examples = load_examples(DATASET_NAME)
n = len(examples) if LIMIT is None else min(LIMIT, len(examples))
rows = [examples[i] for i in range(n)]
probe = load_probe(PROBE_PATH)
scores = score_examples(rows, probe=probe, batch_size=BATCH_SIZE)

# Repeated row IDs are averaged because the scorer expects one prediction per ID.
submission = (
    pd.DataFrame({"index": [row["index"] for row in rows], "score": scores})
    .groupby("index", as_index=False, sort=False)["score"].mean()
)
write_submission(submission["index"], submission["score"])
print(f"scored {n} rows with trusted layer {probe.layer}, pools {probe.pool_widths}")